# Experiment: Per-Product Forecasting dengan Global Model

Notebook ini mengeksplorasi model prediksi per `product_id` menggunakan satu global LightGBM.

**Fitur Kategori Tambahan:**
- `product_model` (Nama model baju)
- `product_color` (Warna)
- `product_size` (Ukuran)

**Target:** Evaluasi metrik (RMSE, MAE) tanpa data dummy terlebih dahulu.

## 1. Setup dan Import

In [ ]:
import os
import sys
import warnings

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sqlalchemy import text

warnings.filterwarnings('ignore')

# Setup path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
from config import engine

print("✓ Semua imports berhasil")

: 

## 2. Query Data per Product_ID

In [ ]:
def fetch_per_product_data() -> pd.DataFrame:
    """
    Mengambil data per product_id dengan detail model, warna, ukuran.
    Hanya mengambil transaksi dengan status SELESAI.
    """
    query = """
        SELECT 
            fa.product_id,
            pr.product_model,
            pr.product_color,
            pr.product_size,
            pr.is_muslim_fashion,
            dd.date AS order_date, 
            fa.total_quantity AS quantity
        FROM fact_daily_agregat fa
        JOIN date_dimension dd ON dd.date_id = fa.date_id
        JOIN product_dimension pr ON pr.product_id = fa.product_id
        WHERE fa.status = 'SELESAI'
        ORDER BY pr.product_id, dd.date
    """
    with engine.connect() as conn:
        df = pd.read_sql(text(query), conn)

    df["order_date"] = pd.to_datetime(df["order_date"])
    
    return df

# Load data
df_raw = fetch_per_product_data()
print(f"Data mentah berhasil dimuat: {len(df_raw)} baris")
print(f"\nTanggal range: {df_raw['order_date'].min()} hingga {df_raw['order_date'].max()}")
print(f"\nJumlah product_id unik: {df_raw['product_id'].nunique()}")
print(f"\nSample data:")
print(df_raw.head(10))

## 3. Explorasi Data Sparsitas

In [ ]:
# Statistik sparsitas
product_counts = df_raw['product_id'].value_counts().sort_values(ascending=False)

print(f"Distribusi jumlah hari per product_id:")
print(f"  Min: {product_counts.min()}")
print(f"  Max: {product_counts.max()}")
print(f"  Mean: {product_counts.mean():.1f}")
print(f"  Median: {product_counts.median():.1f}")

print(f"\nTop 10 produk berdasarkan jumlah hari dengan penjualan:")
print(product_counts.head(10))

print(f"\nDistribusi quantity per product_id:")
qty_by_product = df_raw.groupby('product_id')['quantity'].agg(['sum', 'mean', 'max', 'min'])
print(qty_by_product.head(10))

## 4. Feature Engineering per Product_ID

In [ ]:
def engineer_features_per_product(df: pd.DataFrame) -> pd.DataFrame:
    """
    Feature engineering untuk setiap product_id.
    Lag dan rolling dihitung per product_id.
    """
    # Sort untuk memastikan urutan waktu
    df = df.sort_values(['product_id', 'order_date']).reset_index(drop=True)
    
    # Ambil date features
    query_dates = """
        SELECT 
            date AS order_date, month,
            is_twin_date, is_ramadhan
        FROM date_dimension
        WHERE date BETWEEN '2024-01-01' AND '2026-12-31'
    """
    with engine.connect() as conn:
        df_date_prop = pd.read_sql(text(query_dates), conn)
    
    df_date_prop["order_date"] = pd.to_datetime(df_date_prop["order_date"])
    
    # Merge dengan date dimension
    df = df.merge(df_date_prop, on="order_date", how="left")
    
    # ---- Fitur Rolling dan Lag per product_id ----
    df["rolling_mean_3_qty"] = (
        df.groupby("product_id")["quantity"]
        .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
    )
    
    df["rolling_mean_7_qty"] = (
        df.groupby("product_id")["quantity"]
        .transform(lambda x: x.shift(1).rolling(window=7, min_periods=1).mean())
    )
    
    df["rolling_std_7_qty"] = (
        df.groupby("product_id")["quantity"]
        .transform(lambda x: x.shift(1).rolling(window=7, min_periods=1).std())
    )
    
    # ---- Lag Quantity ----
    df["lag_1_qty"] = df.groupby("product_id")["quantity"].shift(1)
    df["lag_3_qty"] = df.groupby("product_id")["quantity"].shift(3)
    df["lag_7_qty"] = df.groupby("product_id")["quantity"].shift(7)
    
    df["lag_7_rolling_mean"] = df.groupby("product_id")["rolling_mean_7_qty"].shift(7)
    
    # ---- Calendar Features ----
    df["day_of_year"] = df["order_date"].dt.dayofyear
    df["is_month_start"] = df["order_date"].dt.is_month_start.astype(int)
    
    # ---- Hilangkan rows dengan NaN ----
    df_clean = df.dropna().reset_index(drop=True)
    
    return df_clean

# Engineer features
df_features = engineer_features_per_product(df_raw.copy())
print(f"Data setelah feature engineering: {len(df_features)} baris")
print(f"Jumlah product_id di training data: {df_features['product_id'].nunique()}")
print(f"\nSample features:")
print(df_features[['product_id', 'product_model', 'product_color', 'order_date', 
                    'quantity', 'lag_1_qty', 'rolling_mean_7_qty']].head(10))

## 5. Persiapan Features dan Target

In [ ]:
# Target
target = df_features["quantity"].copy()

# Features
features_cols = [
    "product_model", "product_color", "product_size",  # Kategori baru
    "is_muslim_fashion",
    "month",
    "is_twin_date", "is_ramadhan",
    "rolling_mean_3_qty", "rolling_mean_7_qty", "rolling_std_7_qty",
    "lag_1_qty", "lag_3_qty", "lag_7_qty", "lag_7_rolling_mean",
    "day_of_year", "is_month_start"
]

X = df_features[features_cols].copy()

print(f"Target shape: {target.shape}")
print(f"Features shape: {X.shape}")
print(f"\nFeature list: {features_cols}")
print(f"\nFeature dtypes:")
print(X.dtypes)

## 6. Train-Test Split (80-20)

In [ ]:
split_index = int(len(df_features) * 0.8)

X_train, X_test = X.iloc[:split_index].copy(), X.iloc[split_index:].copy()
y_train, y_test = target.iloc[:split_index].copy(), target.iloc[split_index:].copy()

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTrain quantity stats:")
print(y_train.describe())
print(f"\nTest quantity stats:")
print(y_test.describe())

## 7. Kategorisasi Fitur

In [ ]:
# Fitur kategori
categorical_features = ["product_model", "product_color", "product_size", "month"]

# Convert to category
for col in categorical_features:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)
    X_train[col] = X_train[col].astype("category")
    X_test[col] = pd.Categorical(
        X_test[col], 
        categories=X_train[col].cat.categories
    )

print(f"Fitur kategori berhasil dikonversi:")
for col in categorical_features:
    print(f"  {col}: {X_train[col].nunique()} kategori")

## 8. Grid Search dengan TimeSeriesSplit

In [ ]:
# Fixed parameters (sama seperti retrain.py)
fixed_params = {
    "n_estimators": 100,
    "learning_rate": 0.1,
    "random_state": 42,
    "verbose": -1,
}

# Base model
model_base = lgb.LGBMRegressor(**fixed_params)

# Parameter grid
param_grid = [
    {"max_depth": [3], "num_leaves": [5, 7], "min_child_samples": [10, 15, 20]},
    {"max_depth": [4, 5], "num_leaves": [7, 10, 15], "min_child_samples": [10, 15, 20]},
]

# Time Series Split
tscv = TimeSeriesSplit(n_splits=3)

# Grid Search
print("Memulai GridSearchCV dengan TimeSeriesSplit...")
grid_search = GridSearchCV(
    estimator=model_base,
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    refit=False,
    verbose=1,
)

grid_search.fit(X_train, y_train)
challenger_params = grid_search.best_params_

print(f"\n✓ Grid Search selesai")
print(f"Best parameters: {challenger_params}")
print(f"Best CV score (neg RMSE): {grid_search.best_score_:.4f}")

## 9. Training Challenger Model dengan Early Stopping

In [ ]:
# Validation split
val_size = int(len(X_train) * 0.15)
X_tr, X_val = X_train.iloc[:-val_size], X_train.iloc[-val_size:]
y_tr, y_val = y_train.iloc[:-val_size], y_train.iloc[-val_size:]

print(f"Train: {X_tr.shape}, Val: {X_val.shape}")

# Training dengan early stopping
print("\nTraining challenger model dengan early stopping...")
challenger_model = lgb.LGBMRegressor(**fixed_params, **challenger_params)
challenger_model.fit(
    X_tr,
    y_tr,
    eval_set=[(X_val, y_val), (X_train, y_train)],
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=False)],
)

print(f"✓ Best iteration: {challenger_model.best_iteration_}")

# Retrain dengan best iteration
best_n_estimators = challenger_model.best_iteration_ or fixed_params["n_estimators"]
challenger_model = lgb.LGBMRegressor(
    **{**fixed_params, **challenger_params, "n_estimators": best_n_estimators}
)
challenger_model.fit(X_train, y_train, eval_metric="rmse")
print(f"✓ Final model trained dengan {best_n_estimators} estimators")

## 10. Evaluasi pada Test Set

In [ ]:
# Prediksi
y_pred_challenger = np.maximum(challenger_model.predict(X_test), 0)

# Metrics
challenger_rmse = np.sqrt(mean_squared_error(y_test, y_pred_challenger))
challenger_mae = mean_absolute_error(y_test, y_pred_challenger)
challenger_mape = np.mean(np.abs((y_test - y_pred_challenger) / (y_test + 1))) * 100

print("="*60)
print("EVALUASI CHALLENGER MODEL (Per-Product dengan Kategori)")
print("="*60)
print(f"RMSE:  {challenger_rmse:.4f}")
print(f"MAE:   {challenger_mae:.4f}")
print(f"MAPE:  {challenger_mape:.4f}%")
print("="*60)

## 11. Feature Importance

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': features_cols,
    'importance': challenger_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 15 Feature Importance:")
print(feature_importance.head(15))

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance.head(15), x='importance', y='feature', palette='viridis')
plt.title('Top 15 Feature Importance - Per-Product Model')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 12. Analisis Error

In [ ]:
# Error analysis
error_df = pd.DataFrame({
    'actual': y_test.values,
    'predicted': y_pred_challenger,
    'error': np.abs(y_test.values - y_pred_challenger),
    'error_pct': np.abs(y_test.values - y_pred_challenger) / (y_test.values + 1) * 100
})

print("\nError Statistics:")
print(error_df['error'].describe())
print(f"\nError Percentile:")
print(f"  50th: {error_df['error'].quantile(0.5):.2f}")
print(f"  75th: {error_df['error'].quantile(0.75):.2f}")
print(f"  90th: {error_df['error'].quantile(0.90):.2f}")
print(f"  95th: {error_df['error'].quantile(0.95):.2f}")

# Distribusi error
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(error_df['error'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Distribusi Absolute Error')
axes[0].set_xlabel('Error')
axes[0].set_ylabel('Frequency')

axes[1].scatter(error_df['actual'], error_df['predicted'], alpha=0.5)
axes[1].plot([error_df['actual'].min(), error_df['actual'].max()], 
             [error_df['actual'].min(), error_df['actual'].max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Quantity')
axes[1].set_ylabel('Predicted Quantity')
axes[1].set_title('Actual vs Predicted')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 13. Breakdown Metrik per Kelompok

In [ ]:
# Tambahkan product info ke error_df
test_indices = X_test.index.tolist()
error_df['product_model'] = df_features.loc[test_indices, 'product_model'].values
error_df['product_color'] = df_features.loc[test_indices, 'product_color'].values
error_df['quantity_bin'] = pd.cut(error_df['actual'], 
                                   bins=[0, 1, 3, 5, float('inf')],
                                   labels=['0', '1-2', '3-4', '5+'])

# RMSE per model
print("\nRMSE per Product Model (Top 10):")
rmse_per_model = error_df.groupby('product_model').apply(
    lambda g: np.sqrt(mean_squared_error(g['actual'], g['predicted']))
).sort_values(ascending=False)
print(rmse_per_model.head(10))

# MAE per quantity range
print("\nMAE per Quantity Range:")
mae_per_qty_range = error_df.groupby('quantity_bin')['error'].agg(['mean', 'std', 'count'])
print(mae_per_qty_range)

## 14. Kesimpulan dan Rekomendasi

In [ ]:
print("\n" + "="*70)
print("SUMMARY: PER-PRODUCT FORECASTING DENGAN GLOBAL MODEL")
print("="*70)

print(f"""
📊 KONFIGURASI:
  - Target: Quantity per product_id per order_date
  - Model: LightGBM (Global untuk semua variasi)
  - Fitur Kategori: product_model, product_color, product_size, month
  - Total Features: {len(features_cols)}
  - Training Data: {len(X_train)} baris
  - Test Data: {len(X_test)} baris

📈 HASIL METRIK:
  - RMSE:  {challenger_rmse:.4f} pcs
  - MAE:   {challenger_mae:.4f} pcs
  - MAPE:  {challenger_mape:.4f}%

🔍 OBSERVASI:
  - Jumlah unique product_id: {df_features['product_id'].nunique()}
  - Min hari per produk: {product_counts.min()}
  - Max hari per produk: {product_counts.max()}
  - Mean hari per produk: {product_counts.mean():.1f}
  - Data sparsity: {(error_df['actual'] == 0).sum() / len(error_df) * 100:.1f}% adalah 0

💡 NEXT STEPS:
  1. Analisis lebih lanjut pada produk dengan high error
  2. Pertimbangkan hierarchical fallback untuk produk dengan histori sedikit
  3. Implementasi zero-filled panel untuk meningkatkan coverage
  4. Bandingkan dengan model dua tahap (binary + regression)
  5. Evaluasi pada periode forecasting future untuk validasi
""")
print("="*70)